In [7]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib


In [9]:
hs = pd.read_csv("../data/global_house_purchase_dataset.csv")
hs.head()

,property_id,country,city,property_type,furnishing_status,property_size_sqft,price,constructed_year,previous_owners,rooms,...,customer_salary,loan_amount,loan_tenure_years,monthly_expenses,down_payment,emi_to_income_ratio,satisfaction_score,neighbourhood_rating,connectivity_score,decision
0,1,France,Marseille,Farmhouse,Semi-Furnished,991,412935,1989,6,6,...,10745,193949,15,6545,218986,0.16,1,5,6,0
1,2,South Africa,Cape Town,Apartment,Semi-Furnished,1244,224538,1990,4,8,...,16970,181465,20,8605,43073,0.08,9,1,2,0
2,3,South Africa,Johannesburg,Farmhouse,Semi-Furnished,4152,745104,2019,5,2,...,21914,307953,30,2510,437151,0.09,6,8,1,0
3,4,Germany,Frankfurt,Farmhouse,Semi-Furnished,3714,1110959,2008,1,3,...,17980,674720,15,8805,436239,0.33,2,6,6,0
4,5,South Africa,Johannesburg,Townhouse,Fully-Furnished,531,99041,2007,6,3,...,17676,65833,25,8965,33208,0.03,3,3,4,0


La nueva idea es crear nuevas variables a partir de las originales para que el modelo pueda aprender patrones que con los numeros crudos no podia vería 

In [ ]:
# Trabajamos sobre el DataFrame original cargado al inicio
hs_processed = hs.copy()

# Transformaciones logarítmicas para estabilizar la varianza y mejorar la normalidad
hs_processed["log_price"] = np.log(hs_processed["price"])
hs_processed["log_customer_salary"] = np.log(hs_processed["customer_salary"])
hs_processed["log_monthly_expenses"] = np.log(hs_processed["monthly_expenses"])
hs_processed["log_property_size_sqft"] = np.log(hs_processed["property_size_sqft"])

# Media de cuanto puede pagar el cliente en relacion al precio de la casa
hs_processed["affordability_ratio"] = hs_processed["log_customer_salary"] - hs_processed["log_price"]
# Carga financiera del cliente en relación a sus ingresos
hs_processed["expense_burden"] = hs_processed["log_monthly_expenses"] - hs_processed["log_customer_salary"]
# Relación entre el monto del préstamo y el precio de la casa
hs_processed["loan_to_price"] = hs_processed["loan_amount"] / hs_processed["price"]
# Precio por pie cuadrado de la propiedad
hs_processed["price_per_sqft"] = hs_processed["price"] / hs_processed["property_size_sqft"]

# Reemplazar inf y -inf con NaN, luego llenar con la mediana
hs_processed["loan_to_price"].replace([np.inf, -np.inf], np.nan, inplace=True)
hs_processed["price_per_sqft"].replace([np.inf, -np.inf], np.nan, inplace=True)
median_loan = hs_processed["loan_to_price"].median()
median_price_sqft = hs_processed["price_per_sqft"].median()
hs_processed["loan_to_price"].fillna(median_loan, inplace=True)
hs_processed["price_per_sqft"].fillna(median_price_sqft, inplace=True)

/tmp/ipykernel_318966/2430566136.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  hs_processed["loan_to_price"].replace([np.inf, -np.inf], np.nan, inplace=True)
/tmp/ipykernel_318966/2430566136.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col

(70%/20%/10%)

In [ ]:
# Definimos las características correctas 
cat_features = ["country", "city", "property_type", "furnishing_status"]

num_features = [
    "satisfaction_score", "crime_cases_reported", "legal_cases_on_property",
    "constructed_year", "previous_owners", "rooms", "bathrooms", "garage", "garden",
    "loan_tenure_years", "emi_to_income_ratio", "down_payment", "loan_amount",
    "log_property_size_sqft", "log_price", "log_customer_salary", "log_monthly_expenses",
    "affordability_ratio", "expense_burden", "loan_to_price", "price_per_sqft"
]

# Filtramos para solo incluir columnas existentes 
cat_features = [col for col in cat_features if col in hs_processed.columns]
num_features = [col for col in num_features if col in hs_processed.columns]


from sklearn.model_selection import train_test_split

X = hs_processed[cat_features + num_features]
y = hs_processed["decision"]

SEED = 42
# 10% para validación
X_temp, X_val, y_temp, y_val = train_test_split(
    X, y, test_size=0.10, random_state=SEED, stratify=y
)
#20% para test 
X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp, test_size=0.2222, random_state=SEED, stratify=y_temp
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Val: {X_val.shape}")

Train: (140004, 25), Test: (39996, 25), Val: (20000, 25)


In [26]:


# Crear el preprocesador
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ("num", StandardScaler(), num_features)
])

# Ajustar SOLO con train
X_train_processed = preprocessor.fit_transform(X_train)
# Transformar test y val (sin hacer fit)
X_test_processed = preprocessor.transform(X_test)
X_val_processed = preprocessor.transform(X_val)

# Convertir a float32 para ahorrar memoria 
X_train_processed = X_train_processed.astype('float32')
X_test_processed  = X_test_processed.astype('float32')
X_val_processed   = X_val_processed.astype('float32')

print(f"Dimensiones tras preprocesamiento:")
print(f"Train: {X_train_processed.shape}")
print(f"Test:  {X_test_processed.shape}")
print(f"Val:   {X_val_processed.shape}")
print(f"\nNúmero total de características: {X_train_processed.shape[1]}")

Dimensiones tras preprocesamiento:
Train: (140004, 83)
Test:  (39996, 83)
Val:   (20000, 83)

Número total de características: 83


In [27]:
# Guardamos los arrays comprimidos
np.savez_compressed("../data/train_processed.npz", X=X_train_processed, y=y_train.to_numpy(dtype='float32'))
np.savez_compressed("../data/test_processed.npz", X=X_test_processed, y=y_test.to_numpy(dtype='float32'))
np.savez_compressed("../data/val_processed.npz", X=X_val_processed, y=y_val.to_numpy(dtype='float32'))

# Guardamos el preprocesador para reutilizarlo
joblib.dump(preprocessor, "../models/preprocessor.pkl")

print("¡Datos guardados exitosamente!")
print("Validación guardada bajo llave. No la leeremos hasta el final.")

¡Datos guardados exitosamente!
Validación guardada bajo llave. No la leeremos hasta el final.
